# T03 · The projected Hessian loss, derived and verified

**openQHA theory notebook.** This is the long form of `.scratch/hessian-learning-set/design-phl-loss.md`
(approved 2026-09-18): every equation of the loss that fine-tunes MACE-OFF23 on reference
Hessians is *derived* here from the Taylor expansion of the energy, and every identity is
*checked numerically* in the cell that follows it — on the propanal fixture
(`tests/data/propanal_molecule/`: the wB97M-D3(BJ)/def2-TZVPPD analytic Hessian from ORCA and
the stored MACE-OFF23_medium Hessian at the same geometry) and, for the autograd parts, on a
small torch potential with parameters and, if the model loads, on MACE itself.

Reading order: §1 harmonic analysis (why mass weighting, why projection, where the gradient
term lives) → §2 the target loss and its three properties → §3 mode weighting →
§4 the Hutchinson estimator and its variance → §5 the Hessian–vector product by autograd →
§6 the full loss as a deterministic probe set → §7 the gradient and the cost → §8 the batch
identity → §9 algorithm listings → §10 the acceptance table. Nothing here is a claim without
a cell.

Conventions: $N$ atoms, $x\in\mathbb R^{3N}$, energies in eV, lengths in Å, masses in amu;
$\|\cdot\|_F$ the Frobenius norm, $\|\cdot\|_2$ the spectral norm; $\operatorname{tr}$ the trace.


In [1]:
import sys, math, time
from pathlib import Path
import numpy as np

def _repo_root():
    for p in [Path.cwd()] + list(Path.cwd().parents):
        if (p / "openqha" / "__init__.py").is_file():
            return p
    raise RuntimeError("run from inside the openQHA checkout")

ROOT = _repo_root(); sys.path.insert(0, str(ROOT))
from ase.data import atomic_masses, atomic_numbers
from openqha.qm_interfaces import orca
from openqha.thermochem import hessian as hessian_mod, hessian_compare as hc, thermo

FIX = ROOT / "tests" / "data" / "propanal_molecule"
LEVEL = "wb97m-d3bj_def2-tzvppd"
parsed = orca.parse_hess(FIX / "msrrho" / f"orca.{LEVEL}.basin00.hess")
symbols = list(parsed["symbols"])
x_r = np.asarray(parsed["positions_bohr"]) / orca.BOHR_PER_ANGSTROM          # [N, 3] Å, the wB97M minimum
masses = np.array([atomic_masses[atomic_numbers[s]] for s in symbols])       # amu
H_r = orca.hessian_to_ev_per_angstrom2(parsed["hessian_eh_bohr2"])           # reference Cartesian Hessian, eV/Å²
H_t = np.load(FIX / "mace" / "basin00" / f"hessian_at_{LEVEL}.npy")           # MACE-OFF23_medium at the same x_r
N = len(symbols); n3 = 3 * N
print(f"{''.join(symbols)}: N = {N}, 3N = {n3}; H_r {H_r.shape} symmetric to {np.abs(H_r - H_r.T).max():.1e}, "
      f"H_theta symmetric to {np.abs(H_t - H_t.T).max():.1e} eV/Å²")

CCCOHHHHHH: N = 10, 3N = 30; H_r (30, 30) symmetric to 0.0e+00, H_theta symmetric to 0.0e+00 eV/Å²


## 1. Harmonic analysis from the Taylor expansion — mass weighting, projection, and the gradient term

Around a geometry $x_0$ the energy of any level is

$$E(x_0+\delta)=E(x_0)+g^{\!\top}\delta+\tfrac12\,\delta^{\!\top}H\,\delta+O(\delta^3),\qquad g=\nabla E,\ H=\nabla^2E .$$

**Mass weighting.** Newton's equations $M\ddot\delta=-H\delta$ (with $g=0$) become, in
$q=M^{1/2}\delta$, $\ddot q=-Kq$ with $K=M^{-1/2}HM^{-1/2}$: the eigenvalues of $K$ are the
squared angular frequencies. So the object whose spectrum is thermochemistry is $K$, not $H$
— every error measure below is on $K$.

**Rigid motions.** Translation invariance $E(x+\varepsilon\,\mathbf 1_\alpha)=E(x)$ for all
$\varepsilon$ gives, differentiating twice at $\varepsilon=0$, $\mathbf 1_\alpha^{\!\top}H\,\mathbf 1_\alpha=0$
and in fact $H\mathbf 1_\alpha=0$ (differentiate $\nabla E(x+\varepsilon\mathbf 1_\alpha)=\nabla E(x)$
once): the three translations are in the kernel of $H$ **at every geometry**. Rotation
invariance $E(R(\varepsilon)x)=E(x)$ gives, with $x(\varepsilon)=R(\varepsilon)x$,
$$\frac{d^2}{d\varepsilon^2}E(x(\varepsilon))\Big|_0 = r^{\!\top}Hr + g^{\!\top}\frac{d^2x}{d\varepsilon^2}\Big|_0 = 0,\qquad r=\frac{dx}{d\varepsilon}\Big|_0 ,$$
so $r^{\!\top}Hr=-g^{\!\top}x''(0)\neq0$ unless $g=0$: **at a non-stationary geometry the
rotational block of $H$ is not zero, and its size is set by the gradient.** This is the
"gradient term" measured on 2026-09-17 (S0-C-43): 29 cm$^{-1}$ on the tightly converged
propanal minimum, ~100 at a foreign geometry. In mass-weighted coordinates the rigid vectors are
$t_\alpha\propto M^{1/2}\mathbf 1_\alpha$ and $r_\alpha\propto M^{1/2}\,(e_\alpha\times(x_i-x_{\rm com}))_i$,
orthonormalised into $V\in\mathbb R^{3N\times n_{\rm rig}}$ ($n_{\rm rig}=6$, 5 if linear).

**Projection.** $P=I-VV^{\!\top}$ is symmetric and idempotent ($P^2=P$, $PV=0$). The projected
matrix $\tilde K=PKP$ has the rigid subspace in its kernel *by construction* and its other
$n_{\rm vib}=3N-n_{\rm rig}$ eigenpairs $(\lambda_i,L_i)$ are the vibrations:
$\omega_i=\operatorname{sign}(\lambda_i)\sqrt{|\lambda_i|}$.

**Units.** $\lambda$ in eV Å$^{-2}$ amu$^{-1}$; $\omega=\sqrt{\lambda\,e/(\text{amu}\,\text{Å}^2)}$ rad s$^{-1}$;
$\tilde\nu=\omega/(2\pi c)$ cm$^{-1}$. The constant is `hessian.CM_INV_PER_SQRT_EV_A2_AMU`.


In [2]:
m3 = np.repeat(masses, 3)
Msqrt_inv = 1.0 / np.sqrt(m3)
K_r = H_r / np.sqrt(np.outer(m3, m3)); K_t = H_t / np.sqrt(np.outer(m3, m3))
V, sing, rank = hessian_mod.rigid_body_vectors(masses, x_r)
P = np.eye(n3) - V @ V.T
pr = hc.projected(H_r, masses, x_r); pt = hc.projected(H_t, masses, x_r)
L_r, lam_r, omega_r = pr["vec"], pr["lam"], pr["freq"]          # reference modes, eigenvalues, cm^-1
n_vib = L_r.shape[1]
to_cm = lambda lam: np.sign(lam) * np.sqrt(np.abs(lam)) * hessian_mod.CM_INV_PER_SQRT_EV_A2_AMU

# unit constant from SI
e, amu, A, c = 1.602176634e-19, 1.66053906660e-27, 1e-10, 2.99792458e10
print("cm^-1 per sqrt(eV/Å²/amu):", hessian_mod.CM_INV_PER_SQRT_EV_A2_AMU, "vs SI", math.sqrt(e / (amu * A * A)) / (2 * math.pi * c))
print("rank of the rigid subspace:", rank, "| P symmetric", np.abs(P - P.T).max() < 1e-12, "| P idempotent", np.abs(P @ P - P).max() < 1e-12, "| P V = 0:", np.abs(P @ V).max() < 1e-12)
# the gradient term: rigid block of the UNPROJECTED K, translations vs rotations
T = np.zeros((n3, 3))
for a in range(3): T[a::3, a] = np.sqrt(m3[a::3])
T /= np.linalg.norm(T, axis=0)
print("translation block of K_r, cm^-1:", np.round(to_cm(np.linalg.eigvalsh(T.T @ K_r @ T)), 3), " <- H 1 = 0 at every geometry")
print("full rigid block of K_r,  cm^-1:", np.round(to_cm(np.linalg.eigvalsh(V.T @ K_r @ V)), 1), " <- rotations feel the residual gradient")
print("rigid block of the projected K~_r:", np.abs(V.T @ pr["K"] @ V).max(), " <- zero by construction; says nothing")
print(f"n_vib = {n_vib}; lowest reference modes cm^-1: {np.round(omega_r[:4], 1)}; lowest MACE modes at x_r: {np.round(pt['freq'][:4], 1)}")

cm^-1 per sqrt(eV/Å²/amu): 521.4708983725064 vs SI 521.4708983725064
rank of the rigid subspace: 6 | P symmetric True | P idempotent True | P V = 0: True
translation block of K_r, cm^-1: [-0.003 -0.001  0.003]  <- H 1 = 0 at every geometry
full rigid block of K_r,  cm^-1: [-29.  -16.  -13.2   1.4   2.2   4.6]  <- rotations feel the residual gradient
rigid block of the projected K~_r: 1.0034512541691028e-15  <- zero by construction; says nothing
n_vib = 24; lowest reference modes cm^-1: [131.1 222.2 256.2 667.3]; lowest MACE modes at x_r: [127.  228.7 254.  654.9]


## 2. The target loss and its three properties

With $A_\theta=P\,M^{-1/2}(H_\theta-H_r)\,M^{-1/2}P=\tilde K_\theta-\tilde K_r$ (symmetric, $A_\theta V=0$):

$$L_H(\theta)=\frac1{n_{\rm vib}}\|A_\theta\|_F^2 .\tag{1}$$

**(i) Zero iff spectrum and modes agree.** $\|A\|_F=0\iff A=0\iff\tilde K_\theta=\tilde K_r$;
two symmetric matrices are equal iff they have the same eigen-decomposition, i.e. the same
eigenvalues *and* the same eigenvectors (up to degenerate rotations). A loss on sorted
frequencies alone cannot see a wrong mode; (1) does.

**(ii) Every frequency follows (Weyl).** For symmetric $B$, $C$ and eigenvalues sorted
ascending, Courant–Fischer gives
$\lambda_i(B+C)=\min_{\dim S=i}\max_{u\in S,\|u\|=1}u^{\!\top}(B+C)u$, and
$u^{\!\top}Cu\le\|C\|_2$ for every unit $u$, so
$\lambda_i(B)-\|C\|_2\le\lambda_i(B+C)\le\lambda_i(B)+\|C\|_2$. With $B=\tilde K_r$, $C=A_\theta$:

$$|\lambda_i^\theta-\lambda_i^r|\le\|A_\theta\|_2\le\|A_\theta\|_F .$$

So driving (1) to zero drives every eigenvalue, hence every frequency, to the reference; the
bound is what makes (1) a *sufficient* control of the spectrum.

**(iii) The loss in the reference-mode basis is `hessian_compare` family 4.** $L_r$ has
orthonormal columns spanning $\operatorname{range}(P)$, so $L_rL_r^{\!\top}=P$. Because
$A=AP=PA$,
$$\|A\|_F^2=\operatorname{tr}(A^{\!\top}\!A)=\operatorname{tr}(A\,P\,A)=\operatorname{tr}(A\,L_rL_r^{\!\top}A)=\|A L_r\|_F^2 ,$$
and $AL_r=P(AL_r)=L_r(L_r^{\!\top}AL_r)$ (expand $AL_r$ in the basis $L_r$; the $(I-P)$
component is zero), so $\|AL_r\|_F^2=\|L_r^{\!\top}AL_r\|_F^2$ since $L_r$ is an isometry.
Finally $L_r^{\!\top}AL_r=L_r^{\!\top}\tilde K_\theta L_r-L_r^{\!\top}\tilde K_rL_r=D-\Lambda_r$ with
$D=L_r^{\!\top}\tilde K_\theta L_r$. Hence

$$\|A\|_F^2=\|D-\Lambda_r\|_F^2=\sum_i\big(D_{ii}-\lambda_i^r\big)^2+\sum_{i\neq j}D_{ij}^2 .\tag{2}$$

The diagonal is the per-mode curvature error `OMEGA_ALONG_REF_CM` vs `OMEGA_REF_CM`, the
off-diagonal is `MIXING`: **ruler and loss are one quantity read two ways.**


In [3]:
A = P @ (K_t - K_r) @ P
A = 0.5 * (A + A.T)
fro2 = np.sum(A * A)
print(f"‖A‖_F² = {fro2:.6e} eV²/Å⁴/amu²  ->  L_H = {fro2 / n_vib:.6e};  A V = 0: {np.abs(A @ V).max():.1e}")
# (ii) Weyl on the sorted vibrational eigenvalues
lam_t = pt["lam"]
weyl_lhs = np.abs(lam_t - lam_r).max(); spec = np.linalg.norm(A, 2); fro = math.sqrt(fro2)
print(f"(ii) max_i |λ_i^θ − λ_i^r| = {weyl_lhs:.4e}  ≤  ‖A‖_2 = {spec:.4e}  ≤  ‖A‖_F = {fro:.4e}  ->  {weyl_lhs <= spec <= fro}")
# (iii) eq. 2
D = L_r.T @ pt["K"] @ L_r
rhs = np.sum((np.diag(D) - lam_r) ** 2) + np.sum((D - np.diag(np.diag(D))) ** 2)
print(f"(iii) ‖A‖_F² = {fro2:.10e}   Σ(D_ii−λ_i)² + Σ_{{i≠j}} D_ij² = {rhs:.10e}   |diff| = {abs(fro2 - rhs):.1e}")
cmp = hc.compare_hessians(H_t, H_r, masses, x_r)
d_along = np.array(cmp["OMEGA_ALONG_REF_CM"]); mixing = cmp["MIXING"]
print(f"      hessian_compare: OMEGA_ALONG_REF_CM² == D_ii to {np.abs(np.sign(d_along) * (d_along / hessian_mod.CM_INV_PER_SQRT_EV_A2_AMU) ** 2 - np.diag(D)).max():.1e};"
      f"  MIXING = {mixing:.4f} == ‖D − diag D‖_F/‖D‖_F = {np.linalg.norm(D - np.diag(np.diag(D))) / np.linalg.norm(D):.4f}")
print("\nper-mode contributions to (2), lowest six reference modes:")
print("  mode  ω_r/cm⁻¹  D_ii→cm⁻¹  (D_ii−λ_i)²   Σ_{j≠i} D_ij²")
for i in range(6):
    print(f"  {i:4d}  {omega_r[i]:8.1f}  {d_along[i]:9.1f}  {(D[i, i] - lam_r[i]) ** 2:.3e}   {np.sum(D[i, :] ** 2) - D[i, i] ** 2:.3e}")

‖A‖_F² = 2.967534e-01 eV²/Å⁴/amu²  ->  L_H = 1.236472e-02;  A V = 0: 1.6e-17
(ii) max_i |λ_i^θ − λ_i^r| = 3.4267e-01  ≤  ‖A‖_2 = 3.6970e-01  ≤  ‖A‖_F = 5.4475e-01  ->  True
(iii) ‖A‖_F² = 2.9675338566e-01   Σ(D_ii−λ_i)² + Σ_{i≠j} D_ij² = 2.9675338566e-01   |diff| = 1.7e-14
      hessian_compare: OMEGA_ALONG_REF_CM² == D_ii to 7.1e-15;  MIXING = 0.0035 == ‖D − diag D‖_F/‖D‖_F = 0.0035

per-mode contributions to (2), lowest six reference modes:
  mode  ω_r/cm⁻¹  D_ii→cm⁻¹  (D_ii−λ_i)²   Σ_{j≠i} D_ij²
     0     131.1      129.4  2.899e-06   1.157e-03
     1     222.2      227.6  8.061e-05   1.407e-03
     2     256.2      254.0  1.672e-05   3.497e-04
     3     667.3      654.9  3.609e-03   3.697e-03
     4     671.6      673.6  9.706e-05   2.073e-03
     5     867.6      870.8  4.184e-04   2.444e-03


## 3. Mode weighting: the entropy's own sensitivity

Let $W=\operatorname{diag}(w_i)$, $w_i\ge0$, one weight per reference mode, and
$P_W=L_rW^{1/2}L_r^{\!\top}$ (symmetric; $P_I=P$ because $L_rL_r^{\!\top}=P$). Replace $P$ by
$P_W$ on both sides of the error operator, $A_W=P_W\,M^{-1/2}\Delta H\,M^{-1/2}P_W$. Using
$L_r^{\!\top}P_W=W^{1/2}L_r^{\!\top}$ (orthonormal columns),
$$L_r^{\!\top}A_WL_r=W^{1/2}\,L_r^{\!\top}M^{-1/2}\Delta H M^{-1/2}L_r\,W^{1/2}=W^{1/2}(D-\Lambda_r)W^{1/2},$$
and the same trace argument as in (iii) (now $A_W=A_WP=PA_W$ still holds since
$\operatorname{range}(P_W)\subseteq\operatorname{range}(P)$) gives

$$L_H^W=\frac1{n_{\rm vib}}\|A_W\|_F^2=\frac1{n_{\rm vib}}\sum_{i,j}w_iw_j\big(D_{ij}-\lambda_i^r\delta_{ij}\big)^2 .\tag{3}$$

**Which weights.** The entropy of one harmonic mode at temperature $T$, $u=hc\tilde\nu/k_BT$:
$S_{\rm HO}(u)=R\big[\tfrac{u}{e^u-1}-\ln(1-e^{-u})\big]$. Differentiating,
$$\frac{d}{du}\frac{u}{e^u-1}=\frac{e^u-1-ue^u}{(e^u-1)^2},\qquad
\frac{d}{du}\big[-\ln(1-e^{-u})\big]=-\frac{e^{-u}}{1-e^{-u}}=-\frac1{e^u-1},$$
$$\frac{dS_{\rm HO}}{du}=R\,\frac{e^u-1-ue^u-(e^u-1)}{(e^u-1)^2}=-R\,\frac{u\,e^u}{(e^u-1)^2}\ \xrightarrow{u\to0}\ -\frac{R}{u},$$
so $|\partial S/\partial\tilde\nu|\propto1/\tilde\nu$ at low frequency: the entropy is a
low-mode quantity, which is where the rings' error sits. msRRHO replaces $S_{\rm HO}$ by
$w(\tilde\nu)S_{\rm HO}+(1-w)S_{\rm FR}$ with $w=1/(1+(\tau/\tilde\nu)^4)$, so the weight is taken
from the *msRRHO* entropy, by finite difference of `thermo.msrrho` with one mode moved — the
interpolation switch and the free-rotor branch are then inside the weight, and the modes
below $\tau$ are not over-driven. Normalise $w_i=|\partial S/\partial\tilde\nu_i|/\max_j|\cdot|$.


In [4]:
def entropy_weights(omega_cm, masses, positions, preset="crest", T=298.15, d=0.5):
    """|dS_msRRHO/d omega_i| by central difference (cal/mol/K per cm^-1), normalised to max 1."""
    base = list(omega_cm); out = []
    for i in range(len(base)):
        up, dn = list(base), list(base); up[i] += d; dn[i] -= d
        s_up = thermo.msrrho(up, masses, positions, preset=preset, temperature_K=T)["S_vib_kcal_per_K"]
        s_dn = thermo.msrrho(dn, masses, positions, preset=preset, temperature_K=T)["S_vib_kcal_per_K"]
        out.append(abs(s_up - s_dn) / (2 * d) * 1000.0)
    out = np.array(out); return out / out.max(), out

w, dS = entropy_weights(omega_r, masses, x_r)
R = 1.987204259e-3 * 1000  # cal/mol/K
u = 1.438776877 * omega_r / 298.15
dS_HO = R * u * np.exp(u) / (np.exp(u) - 1) ** 2 * (1.438776877 / 298.15)   # analytic |dS_HO/dν|, cal/mol/K per cm^-1
print("  ω_r/cm⁻¹   |dS_msRRHO/dν| num   |dS_HO/dν| analytic   w_i")
for i in list(range(6)) + [n_vib - 1]:
    print(f"  {omega_r[i]:8.1f}   {dS[i]:.4e}           {dS_HO[i]:.4e}      {w[i]:.3f}")
P_W = L_r @ np.diag(np.sqrt(w)) @ L_r.T
A_W = P_W @ (K_t - K_r) @ P_W
lhs = np.sum(A_W * A_W) / n_vib
rhs = np.sum(np.outer(w, w) * (D - np.diag(lam_r)) ** 2) / n_vib
print(f"\n(3) ‖A_W‖_F²/n_vib = {lhs:.10e}   Σ w_i w_j (D_ij − λ_i δ_ij)²/n_vib = {rhs:.10e}   |diff| = {abs(lhs - rhs):.1e}")
print(f"    unweighted L_H = {fro2 / n_vib:.4e}; weighted L_H^W = {lhs:.4e} (the low modes carry it); P_I == P: {np.abs(L_r @ L_r.T - P).max():.1e}")

  ω_r/cm⁻¹   |dS_msRRHO/dν| num   |dS_HO/dν| analytic   w_i
     131.1   1.4634e-02           1.4657e-02      1.000
     222.2   8.1325e-03           8.1327e-03      0.556
     256.2   6.8412e-03           6.8411e-03      0.467
     667.3   1.3384e-03           1.3384e-03      0.091
     671.6   1.3171e-03           1.3171e-03      0.090
     867.6   6.2911e-04           6.2910e-04      0.043
    3147.3   3.6916e-08           3.6917e-08      0.000

(3) ‖A_W‖_F²/n_vib = 1.7970472014e-05   Σ w_i w_j (D_ij − λ_i δ_ij)²/n_vib = 1.7970472014e-05   |diff| = 6.4e-19
    unweighted L_H = 1.2365e-02; weighted L_H^W = 1.7970e-05 (the low modes carry it); P_I == P: 2.0e-14


## 4. The Hutchinson estimator (PHL) and its variance

**Unbiasedness.** For any matrix $A\in\mathbb R^{n\times n}$ and a random $v$ with $E[v]=0$,
$E[vv^{\!\top}]=I$ (Rademacher $v_i\in\{-1,+1\}$ equiprobable, or standard Gaussian):
$$\|Av\|^2=\sum_i\Big(\sum_jA_{ij}v_j\Big)^2=\sum_i\sum_{j,k}A_{ij}A_{ik}\,v_jv_k
\ \Rightarrow\ E\|Av\|^2=\sum_i\sum_{j,k}A_{ij}A_{ik}\,\delta_{jk}=\sum_{i,j}A_{ij}^2=\|A\|_F^2 .\tag{4}$$

**Why one draw is one HVP.** With $A=PM^{-1/2}\Delta HM^{-1/2}P$ and $P^2=P$,
$$Av=P\,M^{-1/2}\Delta H\,\underbrace{M^{-1/2}Pv}_{\tilde v}=P\,M^{-1/2}\big(H_\theta\tilde v-H_r\tilde v\big),\qquad \tilde v:=M^{-1/2}Pv .\tag{5}$$
The model side is a Hessian–vector product; the reference side is a matrix–vector product
on the stored label. With $k$ independent probes,
$$\hat L_H^{(k)}=\frac1{n_{\rm vib}k}\sum_{j=1}^k\big\|P\,M^{-1/2}(H_\theta\tilde v_j-H_r\tilde v_j)\big\|^2,\qquad E\big[\hat L_H^{(k)}\big]=L_H\ \ \forall k\ge1 .\tag{6,7}$$

**Variance (Rademacher).** Write $Q=v^{\!\top}Bv$ with $B=A^{\!\top}\!A$ symmetric, so
$\|Av\|^2=Q$. $E[Q]=\operatorname{tr}B$. For the second moment,
$E[Q^2]=\sum_{ijkl}B_{ij}B_{kl}\,E[v_iv_jv_kv_l]$, and for Rademacher variables
$E[v_iv_jv_kv_l]=1$ exactly when the indices pair up ($i=j,k=l$; $i=k,j=l$; $i=l,j=k$; the
all-equal case counted once), else 0. Summing the three pairings and correcting the
double count of $i=j=k=l$:
$$E[Q^2]=(\operatorname{tr}B)^2+2\sum_{i\neq j}B_{ij}^2
\ \Rightarrow\ \operatorname{Var}[Q]=2\sum_{i\neq j}B_{ij}^2=2\big(\|B\|_F^2-\textstyle\sum_iB_{ii}^2\big).$$
(For Gaussian $v$ the fourth moment $E[v_i^4]=3$ adds the diagonal back:
$\operatorname{Var}[Q]=2\|B\|_F^2$, strictly larger — Rademacher is the better probe.)
Averaging $k$ draws and dividing by $n_{\rm vib}$:
$$\operatorname{Var}\big[\hat L_H^{(k)}\big]=\frac{2}{n_{\rm vib}^2\,k}\Big(\|B\|_F^2-\sum_iB_{ii}^2\Big),\qquad B=A^{\!\top}\!A .\tag{8}$$


In [5]:
rng = np.random.default_rng(7)
def one_draw(v):
    vt = Msqrt_inv * (P @ v)                                   # ṽ = M^-1/2 P v   (eq. 5)
    rho = P @ (Msqrt_inv * (H_t @ vt - H_r @ vt))              # P M^-1/2 (H_θ ṽ − H_r ṽ)
    return rho @ rho
n_draw = 4000
rad = np.array([one_draw(rng.choice([-1.0, 1.0], size=n3)) for _ in range(n_draw)])
gau = np.array([one_draw(rng.standard_normal(n3)) for _ in range(n_draw)])
B = A.T @ A
var_rad = 2 * (np.sum(B * B) - np.sum(np.diag(B) ** 2)); var_gau = 2 * np.sum(B * B)
print(f"(4) ‖A‖_F² = {fro2:.6e}")
print(f"    Rademacher mean of ‖Av‖² over {n_draw}: {rad.mean():.6e}  (rel. dev. {abs(rad.mean() / fro2 - 1):.2%});  variance {rad.var():.3e} vs formula {var_rad:.3e}")
print(f"    Gaussian   mean of ‖Av‖² over {n_draw}: {gau.mean():.6e}  (rel. dev. {abs(gau.mean() / fro2 - 1):.2%});  variance {gau.var():.3e} vs formula {var_gau:.3e}")
print("    running mean of the Rademacher estimator / ‖A‖_F² after k draws:",
      {k: round(float(rad[:k].mean() / fro2), 3) for k in (1, 2, 4, 16, 64, 256, 4000)})
print(f"(8) with k = 4 probes: std of L̂_H = {math.sqrt(var_rad / 4) / n_vib:.3e} against L_H = {fro2 / n_vib:.3e}  (relative {math.sqrt(var_rad / 4) / fro2:.2f}) -- one batch of structures averages this down further")

(4) ‖A‖_F² = 2.967534e-01
    Rademacher mean of ‖Av‖² over 4000: 2.950195e-01  (rel. dev. 0.58%);  variance 1.970e-02 vs formula 2.000e-02
    Gaussian   mean of ‖Av‖² over 4000: 2.910763e-01  (rel. dev. 1.91%);  variance 4.109e-02 vs formula 4.360e-02
    running mean of the Rademacher estimator / ‖A‖_F² after k draws: {1: 1.491, 2: 1.122, 4: 1.065, 16: 1.078, 64: 1.058, 256: 1.038, 4000: 0.994}
(8) with k = 4 probes: std of L̂_H = 2.946e-03 against L_H = 1.236e-02  (relative 0.24) -- one batch of structures averages this down further


## 5. The Hessian–vector product by automatic differentiation

Forces are $F_\theta=-\nabla_xE_\theta$. For a probe $\tilde v$ that does not depend on $x$,
$$\big[\nabla_x(F_\theta\!\cdot\!\tilde v)\big]_j=\partial_j\Big(-\sum_i\partial_iE_\theta\,\tilde v_i\Big)=-\sum_i\partial_j\partial_iE_\theta\,\tilde v_i=-(H_\theta\tilde v)_j
\ \Rightarrow\ H_\theta\tilde v=-\nabla_x\big(F_\theta\!\cdot\!\tilde v\big).\tag{9}$$
In autograd: the forces already carry a graph during training (`create_graph=True` in
`compute_forces`), so (9) is **one more backward pass** over that graph, with
`create_graph=True` again so that $H_\theta\tilde v$ stays differentiable with respect to
$\theta$ (a third-order graph, for one vector, not $3N$). The forward-over-reverse form
$\operatorname{jvp}(\nabla_xE_\theta;\tilde v)$ is the same number by the symmetry of $H$.
The shipped `compute_hessians_vmap` builds all $3N$ columns with `create_graph=False` — the
ruler's path, never the loss's.

The cell below checks (9) on a small torch potential with learnable parameters
(a pair term on interatomic distances through a tiny MLP): reverse-over-reverse,
forward-over-reverse, and the explicit Hessian agree; the HVP has a graph to $\theta$.


In [6]:
import torch
torch.manual_seed(0); torch.set_default_dtype(torch.float64)

class ToyPotential(torch.nn.Module):
    """E(x) = Σ_{i<j} f_θ(r_ij) + Σ_i g_θ(|x_i - x_com|²): pair MLP + a confining term, parameters θ."""
    def __init__(self):
        super().__init__()
        self.pair = torch.nn.Sequential(torch.nn.Linear(1, 16), torch.nn.Tanh(), torch.nn.Linear(16, 1))
        self.conf = torch.nn.Parameter(torch.tensor(0.05))
    def forward(self, x, batch=None):                          # x: [n, 3]; batch: [n] graph index (None -> one graph)
        n = x.shape[0]
        idx = torch.zeros(n, dtype=torch.long) if batch is None else batch      # one graph unless told otherwise
        same = (idx[:, None] == idx[None, :]).triu(1)
        diff = x[:, None, :] - x[None, :, :]                     # explicit distances: cdist has no double backward
        d = torch.sqrt((diff ** 2).sum(-1)[same] + 1e-12).unsqueeze(-1)
        e_pair = self.pair(d).sum()
        com = x.mean(0, keepdim=True) if batch is None else torch.stack([x[idx == g].mean(0) for g in idx.unique()])[idx]
        return e_pair + self.conf * ((x - com) ** 2).sum()

toy = ToyPotential()
x = torch.tensor(x_r, requires_grad=True)
E = toy(x)
F = -torch.autograd.grad(E, x, create_graph=True)[0]           # forces with a graph (mace: compute_forces(training=True))
v = torch.randn(N, 3)
hvp_rr = torch.autograd.grad(-(F * v).sum(), x, create_graph=True)[0]          # eq. 9, reverse-over-reverse
H_full = torch.autograd.functional.hessian(lambda p: toy(p), x.detach()).reshape(n3, n3)
hvp_ref = (H_full @ v.reshape(-1)).reshape(N, 3)
grad_fn = lambda p: torch.func.grad(lambda q: toy(q))(p)
hvp_fr = torch.func.jvp(grad_fn, (x.detach(),), (v,))[1]                         # forward-over-reverse
print(f"(9) reverse-over-reverse vs explicit H v: {torch.abs(hvp_rr - hvp_ref).max():.2e}; forward-over-reverse vs explicit: {torch.abs(hvp_fr - hvp_ref).max():.2e}  (A7)")
g_theta = torch.autograd.grad(hvp_rr.pow(2).sum(), list(toy.parameters()), allow_unused=True)
print("    the HVP carries a graph to θ: parameter-gradient norms", {n: round(float(g.norm()), 4) if g is not None else None for (n, _), g in zip(toy.named_parameters(), g_theta)})

(9) reverse-over-reverse vs explicit H v: 6.25e-17; forward-over-reverse vs explicit: 9.71e-17  (A7)
    the HVP carries a graph to θ: parameter-gradient norms {'conf': 9.6707, 'pair.0.weight': 6.4972, 'pair.0.bias': 1.7519, 'pair.2.weight': 14.2236, 'pair.2.bias': None}


## 6. The full E‑F‑H loss is the same estimator with a deterministic probe set

Take the probes to be the reference modes, $v_j=L_{r,j}$, $j=1..n_{\rm vib}$ (orthonormal,
$Pv_j=v_j$). By the cyclic property of the trace and $AP=A$:
$$\sum_{j=1}^{n_{\rm vib}}\|AL_{r,j}\|^2=\operatorname{tr}\big(L_r^{\!\top}A^{\!\top}\!AL_r\big)=\operatorname{tr}\big(A^{\!\top}\!A\,L_rL_r^{\!\top}\big)=\operatorname{tr}\big(A^{\!\top}\!AP\big)=\operatorname{tr}\big(A^{\!\top}\!A\big)=\|A\|_F^2 .\tag{10}$$
The Cartesian unit vectors do the same in $3N$ draws: $\sum_j\|Ae_j\|^2=\sum_j\sum_iA_{ij}^2=\|A\|_F^2$
(the $n_{\rm rig}$ extra probes cost HVPs and contribute zero because $AV=0$ — but $e_j\notin\operatorname{range}(P)$,
so every $e_j$ is needed). So `probe="modes"`, $k=n_{\rm vib}$, **is** the full loss (1) —
exactly, with zero variance — on the same code path as PHL; only $k$ and the probe rule
differ. Term $j$ of the mode sum is, by (2), $\sum_i(D_{ij}-\lambda_i^r\delta_{ij})^2$: column $j$
of family 4.


In [7]:
modes_sum = sum(one_draw(L_r[:, j]) for j in range(n_vib))
cart_sum = sum(one_draw(np.eye(n3)[:, j]) for j in range(n3))
print(f"(10) Σ_j ‖A L_r,j‖² = {modes_sum:.12e}   Σ_j ‖A e_j‖² = {cart_sum:.12e}   ‖A‖_F² = {fro2:.12e}")
print(f"     |modes − Frobenius| = {abs(modes_sum - fro2):.1e}   |cartesian − Frobenius| = {abs(cart_sum - fro2):.1e}   (A2)")
col = np.array([one_draw(L_r[:, j]) for j in range(n_vib)])
print(f"     term j == Σ_i (D_ij − λ_i δ_ij)²: max diff {np.abs(col - np.sum((D - np.diag(lam_r)) ** 2, axis=0)).max():.1e}")
# A3: the projection makes the loss blind to a rigid-block perturbation of the label
eps = (300.0 / hessian_mod.CM_INV_PER_SQRT_EV_A2_AMU) ** 2
H_r_noisy = H_r + eps * np.sqrt(np.outer(m3, m3)) * (V @ V.T)
A_noisy = P @ (K_t - H_r_noisy / np.sqrt(np.outer(m3, m3))) @ P
print(f"(A3) ε V Vᵀ (a 300 cm⁻¹ rigid block) added to the label changes ‖A‖_F² by {abs(np.sum(A_noisy * A_noisy) - fro2):.1e}")
# A5 / A6 on the loss as a pure function
A_same = P @ (K_t - K_t) @ P; A_scaled = P @ (0.81 * K_t - K_t) @ P
print(f"(A5) H_r := H_θ  ->  L_H = {np.sum(A_same * A_same) / n_vib:.1e}")
print(f"(A6) H_r := 0.81 H_θ (0.9x frequencies)  ->  L_H = {np.sum(A_scaled * A_scaled) / n_vib:.6e} == 0.19² ‖K~_θ‖_F²/n_vib = {0.19 ** 2 * np.sum(pt['K'] ** 2) / n_vib:.6e}")

(10) Σ_j ‖A L_r,j‖² = 2.967533856606e-01   Σ_j ‖A e_j‖² = 2.967533856606e-01   ‖A‖_F² = 2.967533856606e-01
     |modes − Frobenius| = 5.6e-16   |cartesian − Frobenius| = 5.6e-17   (A2)
     term j == Σ_i (D_ij − λ_i δ_ij)²: max diff 9.5e-15
(A3) ε V Vᵀ (a 300 cm⁻¹ rigid block) added to the label changes ‖A‖_F² by 6.7e-16
(A5) H_r := H_θ  ->  L_H = 0.0e+00
(A6) H_r := 0.81 H_θ (0.9x frequencies)  ->  L_H = 1.180938e+01 == 0.19² ‖K~_θ‖_F²/n_vib = 1.180938e+01


## 7. The gradient, the cost, and the weight of the Hessian term

**Total loss.** $L(\theta)=w_EL_E+w_FL_F+w_H\hat L_H^{(k)}$ with mace's per-config-weighted
squared errors for $E$ and $F$. Writing $\rho_j=PM^{-1/2}(H_\theta\tilde v_j-H_r\tilde v_j)$
(the reference product does not depend on $\theta$), the chain rule gives
$$\frac{\partial\hat L_H}{\partial\theta}=\frac{2}{n_{\rm vib}k}\sum_j\rho_j^{\!\top}\,P\,M^{-1/2}\,\frac{\partial(H_\theta\tilde v_j)}{\partial\theta},\tag{12}$$
which autograd produces from (9) because the HVP carries a graph: the parameter derivative
of a Hessian–vector product is a third derivative of the energy contracted with $\tilde v$.

**Cost.** Per batch: one forward; one backward for the forces with the graph kept;
$k$ HVP backward passes (each ≈ one backward, graph kept); and one parameter backward
through everything, which re-traverses those tapes. Counting a backward as a forward
equivalent, roughly $(2+2k)$ per batch against $(2+6N)$ for the $3N$ columns of the full
matrix through the same third-order machinery — and, decisively, the memory of one
third-order tape instead of $3N$ simultaneous ones. The measured number is ticket 08's.

**The weight $w_H$.** PHL and Rodriguez report $w_H/w_F\approx0.25$–$0.30$ on *Cartesian,
unweighted* Hessians (eV Å$^{-2}$). Here $K$ carries $1/\sqrt{m_im_j}$, so the same
error is $\sim1/m$ smaller on heavy-atom modes and larger on hydrogenic ones; the band is
a starting point, not a value. The rule: on the base model at epoch 0, set $w_H$ so that
$w_H\hat L_H\approx w_FL_F$ (both measured on the smoke set), then scan $\times0.3$ and
$\times3$ and report all three (plan_C §5.2).

The cell checks (12) numerically — acceptance A4: the autograd gradient of $\hat L_H$ with
respect to one parameter equals its central finite difference — on the toy potential with
the propanal reference label.


In [8]:
H_r_t = torch.tensor(H_r); P_t = torch.tensor(P); Minv_t = torch.tensor(Msqrt_inv)
def phl_loss(model, x_np, probes, H_ref):
    """eq. 6 for one structure: probes [k, 3N] raw v; returns L̂_H (scalar tensor with graph)."""
    x = torch.tensor(x_np, requires_grad=True)
    E = model(x); F = -torch.autograd.grad(E, x, create_graph=True)[0]
    tot = 0.0
    for v in probes:
        vt = (Minv_t * (P_t @ v)).reshape(N, 3)                                   # ṽ
        hvp = torch.autograd.grad(-(F * vt).sum(), x, create_graph=True)[0].reshape(-1)   # H_θ ṽ (eq. 9)
        rho = P_t @ (Minv_t * (hvp - H_ref @ vt.reshape(-1)))                     # P M^-1/2 (H_θ ṽ − H_r ṽ)
        tot = tot + rho @ rho
    return tot / (n_vib * probes.shape[0])

probes = torch.tensor(rng.choice([-1.0, 1.0], size=(4, n3)))
L = phl_loss(toy, x_r, probes, H_r_t)
grads = torch.autograd.grad(L, list(toy.parameters()), allow_unused=True)   # the last bias shifts E only: no Hessian dependence
named = dict(zip([n for n, _ in toy.named_parameters()], grads))
p = toy.pair[0].weight; g_auto = named["pair.0.weight"][3, 0].item()
h = 1e-4
with torch.no_grad(): p[3, 0] += h
L_up = phl_loss(toy, x_r, probes, H_r_t).item()
with torch.no_grad(): p[3, 0] -= 2 * h
L_dn = phl_loss(toy, x_r, probes, H_r_t).item()
with torch.no_grad(): p[3, 0] += h
g_fd = (L_up - L_dn) / (2 * h)
print(f"(12)/A4  L̂_H = {L.item():.6e};  ∂L̂/∂θ autograd = {g_auto:.6e}, central finite difference = {g_fd:.6e}, |diff| = {abs(g_auto - g_fd):.1e}")

(12)/A4  L̂_H = 3.093797e+02;  ∂L̂/∂θ autograd = 4.808792e-01, central finite difference = 4.808792e-01, |diff| = 1.6e-08


## 8. One backward pass per probe for the whole batch

A batch of $B$ structures is $B$ disconnected graphs: $E=\sum_nE_n(x_n)$ with disjoint
variables, so $\partial^2E/\partial x_n\partial x_m=0$ for $n\ne m$ and the batch Hessian is
block-diagonal. Hence, for a batch-wide probe $\tilde v=(\tilde v_1,\dots,\tilde v_B)$,
$$-\nabla_x\Big(\sum_nF_n\!\cdot\!\tilde v_n\Big)=\big(H_1\tilde v_1,\ \dots,\ H_B\tilde v_B\big):$$
the $j$-th probe of every structure comes from **one** backward pass. $k$ probes per
structure cost $k$ passes per batch, not $k\times B$. (This is also why the shipped
`compute_hessians_vmap`, applied to a batch, returns exact zeros off the diagonal blocks.)


In [9]:
x2 = np.vstack([x_r, x_r + np.array([8.0, 0, 0])])                      # two copies, far apart
batch = torch.tensor([0] * N + [1] * N)
xb = torch.tensor(x2, requires_grad=True)
Eb = toy(xb, batch); Fb = -torch.autograd.grad(Eb, xb, create_graph=True)[0]
vb = torch.randn(2 * N, 3)
hvp_b = torch.autograd.grad(-(Fb * vb).sum(), xb)[0]                       # one pass, both structures
hvps = []
for g in (0, 1):
    xg = torch.tensor(x2[g * N:(g + 1) * N], requires_grad=True)
    Fg = -torch.autograd.grad(toy(xg), xg, create_graph=True)[0]
    hvps.append(torch.autograd.grad(-(Fg * vb[g * N:(g + 1) * N]).sum(), xg)[0])
print(f"batch HVP vs per-structure HVPs: max diff {torch.abs(hvp_b - torch.cat(hvps)).max():.1e}  (one backward pass gave both)")

batch HVP vs per-structure HVPs: max diff 1.7e-17  (one backward pass gave both)


## 9. Algorithm listings (the design's Algorithms 1–4, in full)

```
Algorithm 1  probes(x, M, H_r, mode, k, W)                       -- per structure, in the data loader, no autograd
  V  <- rigid_body_vectors(M, x)                                  # 3N x n_rig, orthonormal, mass-weighted   (§1)
  P  <- I - V V^T                                                 # Eckart projector, P^2 = P, P V = 0
  if W is given:  (L_r, Λ_r) <- eigh(P K_r P) on range(P);  P <- L_r W^{1/2} L_r^T            (§3, eq. 3)
  if mode == "rademacher": v_j ~ Uniform{-1,+1}^{3N}, j = 1..k                                  (§4)
  if mode == "gaussian":   v_j ~ N(0, I),               j = 1..k
  if mode == "modes":      v_j <- L_{r,j},              j = 1..n_vib   (k := n_vib; exact, eq. 10)
  if mode == "cartesian":  v_j <- e_j,                  j = 1..3N      (k := 3N;    exact)
  ṽ_j <- M^{-1/2} P v_j                                           # what the model sees            (eq. 5)
  r_j <- P M^{-1/2} H_r ṽ_j                                        # reference side, a matvec
  return {ṽ_j}, {r_j}

Algorithm 2  one training step on a batch B = {n = 1..|B|}
  ({ṽ_{n,j}}, {r_{n,j}}) <- Algorithm 1 for every n                # loader; new Rademacher draws every epoch
  E_θ, F_θ <- model(B, training=True)                              # forces with create_graph=True
  for j = 1..k:                                                    # k backward passes for the WHOLE batch (§8)
      g_j <- autograd.grad( -Σ_n F_{θ,n} · ṽ_{n,j}, x, create_graph=True )   # [H_{θ,n} ṽ_{n,j}]_n   (eq. 9)
      for n:  ρ_{n,j} <- P_n M_n^{-1/2} g_{n,j} - r_{n,j}
  L_H <- (1/|B|) Σ_n (1/(n_vib,n k)) Σ_j ‖ρ_{n,j}‖²                                          (eq. 6)
  L   <- w_E L_E(B) + w_F L_F(B) + w_H L_H                                                    (eq. 11)
  θ   <- optimizer.step(∇_θ L)                                     # third-order graph through the HVPs (eq. 12)
  (forgetting: mace's multihead fine-tuning replays SPICE E/F on the pretraining head in the same loop)

Algorithm 3  the ruler (held-out set)                             -- inference, create_graph=False
  for each structure: H_θ <- MACECalculator.get_hessian(atoms)     # the shipped compute_hessians_vmap
                      hessian_compare.compare_hessians(H_θ, H_r, masses, x)   # families 1-4, floors, D, MIXING
  thermochemistry at every level's own minimum (the harmonic frequency, not the Cartesian curvature)
  in-distribution and out-of-distribution rows separately

Algorithm 4  acceptance (each must be able to fail)
  A1 unbiased    mean of L̂^{(k)} over draws -> eq. 1 within 1 %                  (§4 cell)
  A2 exact       probe = modes / cartesian == eq. 1 to 1e-10                      (§6 cell)
  A3 projection  ε V V^T on H_r changes L̂ by 0                                    (§6 cell)
  A4 gradient    autograd ∂L̂/∂θ == central finite difference                     (§7 cell)
  A5 must-pass   H_r := H_θ  -> L_H = 0                                          (§6 cell)
  A6 must-fail   H_r := 0.81 H_θ -> L_H = 0.19² ‖K~_θ‖_F²/n_vib, ∇ ≠ 0            (§6 cell)
  A7 estimators  reverse-over-reverse == forward-over-reverse HVP                 (§5 cell)
```


## 10. On MACE itself (optional cell)

If MACE-OFF23_medium loads in this environment, the cell runs (9) on the real model: the
force graph from the model's own `compute_forces(training=True)`, one `autograd.grad` for
the HVP, compared with a column of the shipped `get_hessian()` — the check ticket 05 turns
into a test. It is skipped, not failed, when the model is not available.


In [10]:
try:
    from ase import Atoms
    from openqha.potentials import engine
    calc, ename, prov = engine.calculator(device="cpu")
    atoms = Atoms(symbols=symbols, positions=x_r)
    model = calc.models[0]
    b = calc._atoms_to_batch(atoms); bd = calc._clone_batch(b).to_dict()
    out = model(bd, training=True, compute_force=True, compute_stress=False)
    F_m, pos = out["forces"], bd["positions"]
    v_m = torch.zeros_like(pos); v_m.view(-1)[7] = 1.0                        # Cartesian unit probe e_7
    hvp_m = torch.autograd.grad(-(F_m * v_m).sum(), pos)[0].detach().cpu().numpy().reshape(-1)
    H_m = np.asarray(calc.get_hessian(atoms)).reshape(n3, n3)
    print(f"{ename}: HVP by eq. 9 vs column 7 of the shipped get_hessian(): max diff {np.abs(hvp_m - H_m[:, 7]).max():.2e} eV/Å²")
    print(f"the stored fixture Hessian is the same object: max |H_m − H_θ| = {np.abs(H_m - H_t).max():.2e}")
except Exception as exc:                                     # noqa: BLE001
    print("MACE not available here -- skipped:", str(exc)[:160])

cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


/home/ubuntu/anaconda3/envs/openqha/lib/python3.11/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


/mnt/c/Users/10704/Documents/01_Free-Energy-alchemical/openQHA-Hessian/mace/calculators/mace.py:226: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


MACE-OFF23_medium: HVP by eq. 9 vs column 7 of the shipped get_hessian(): max diff 1.42e-14 eV/Å²
the stored fixture Hessian is the same object: max |H_m − H_θ| = 7.11e-15


## Acceptance table

| id | statement | cell | result |
|---|---|---|---|
| A1 | $E[\hat L^{(k)}]=L_H$ (eq. 7) | §4 | mean over 4000 draws within 1 % |
| A2 | mode / Cartesian probes reproduce $\|A\|_F^2$ exactly (eq. 10) | §6 | to 1e-10 |
| A3 | a rigid-block perturbation of the label changes nothing | §6 | 0 |
| A4 | $\partial\hat L/\partial\theta$ autograd = finite difference (eq. 12) | §7 | to 1e-6 |
| A5 | identical-to-base gives $L_H=0$ | §6 | 0 |
| A6 | 0.9× frequencies give $0.19^2\|\tilde K\|_F^2/n_{\rm vib}$ | §6 | equal |
| A7 | reverse-over-reverse = forward-over-reverse HVP (eq. 9) | §5 | to 1e-12 |
| — | eq. 2: loss = family 4 (`OMEGA_ALONG_REF_CM`, `MIXING`) | §2 | to 1e-10 |
| — | eq. 3: weighted loss identity | §3 | to 1e-10 |
| — | eq. 8: Rademacher variance formula; Gaussian larger | §4 | agree |
| — | §8: one backward pass per probe per batch | §8 | to 1e-12 |

References: Hutchinson, *Commun. Stat. Simul.* 18, 1059 (1989); Weyl's inequality via
Courant–Fischer (Horn & Johnson, *Matrix Analysis*, Thm 4.3.1); PHL (arXiv 2026, projected
Hessian learning) and Rodriguez et al., *JCTC* 2025 (full E‑F‑H loss) — `docs/cite/cite_openQHA.bib`;
the entropy derivative is the harmonic-oscillator partition function; msRRHO after Grimme
2012 / Pracht & Grimme 2021 as implemented in `openqha.thermochem.thermo`.
